# Whisper STT evaluation

Compares 8 speech-to-text models on a custom set of 30 UFV campus utterances, all
run on the same Colab GPU for a fair head to head. Each model is tested in two
conditions: no prompt, and with a domain vocabulary prompt. The HuggingFace ASR
models (Wav2Vec2, Speech2Text) do not take a prompt, so their two conditions are
identical by design.

## Models

| Model | Family | Params |
|-|-|-|
| whisper-tiny | Whisper | 39M |
| whisper-base | Whisper | 74M |
| whisper-small | Whisper | 244M |
| whisper-medium | Whisper | 769M |
| whisper-large-v3-turbo | Whisper | 809M |
| whisper-large-v3 | Whisper | 1550M |
| facebook/wav2vec2-base-960h | Wav2Vec2 (CTC) | 95M |
| facebook/s2t-small-librispeech-asr | Speech2Text | 30M |

## Metrics

| Metric | Meaning | Direction |
|-|-|-|
| WER | Word Error Rate | lower is better |
| CER | Character Error Rate | lower is better |
| Sentence Accuracy | fraction transcribed perfectly | higher is better |
| DKA | Domain Keyword Accuracy | higher is better |
| Latency | mean and std over repeated GPU runs | lower is better |
| PIS | Prompt Impact Score (DKA with prompt minus without) | higher is more prompt-dependent |

Decoding is pinned for reproducibility: greedy (`temperature=0.0`, no fallback
ladder), `language=en`, `condition_on_previous_text=False`, seed 42. Repeating the
same (model, condition, audio) gives bit-identical transcriptions, so quality
metrics are computed once and only latency is measured across the repeated runs.

## Outputs

Per-model `results.csv`, the long and wide summary tables, the per-utterance
predictions, the prompt impact scores, the utterance-difficulty ranking, and
`whisper_reproducibility.json`.

## 1. Setup

Mount Google Drive, install dependencies, and verify that the GPU is available.

### 1.1 Mount Google Drive, normalize audios to WAV, verify

Audios must be uploaded to `Drive/MyDrive/experimento_whisper/` as `utterance_01.*` ... `utterance_30.*` in any common container (`.wav`, `.m4a`, `.mp3`, `.flac`, `.ogg`, `.webm`).

This cell (a) mounts Drive, (b) converts any non-WAV utterance to **WAV 16 kHz mono 16-bit PCM** with `ffmpeg` - the exact format required by `whisper.cpp` in the on-device experiment, so both evaluations share the same byte-level dataset - and (c) fails fast if any utterance is missing after normalization. Originals of converted files are moved to `experimento_whisper/originals_backup/`. The conversion step is idempotent; after the first successful pass it is a no-op.

In [ ]:
from google.colab import drive
import os
import re
import shutil
import subprocess
from pathlib import Path

drive.mount('/content/drive')

AUDIO_DIR = '/content/drive/MyDrive/experimento_whisper'
AUDIO_EXTS = ('.wav', '.m4a', '.mp3', '.flac', '.ogg', '.webm')
EXPECTED_STEMS = [f'utterance_{i:02d}' for i in range(1, 31)]

if not os.path.isdir(AUDIO_DIR):
    msg = f'Audio directory not found: {AUDIO_DIR}'
    msg += ' (create it in Drive and upload utterance_01 ... utterance_30 in any audio format)'
    raise FileNotFoundError(msg)

# Preprocess: normalize any non-WAV audios to WAV 16 kHz mono 16-bit PCM
# whisper.cpp (on-device experiment) accepts only this exact format, so converting
# here gives us a single-source-of-truth dataset shared across cloud-GPU and
# on-device evaluations. Originals go to `originals_backup/` instead of being
# deleted. Idempotent: after the first pass there is nothing left to convert.
_audio_dir_path = Path(AUDIO_DIR)
_backup_dir = _audio_dir_path / 'originals_backup'
_backup_dir.mkdir(exist_ok=True)
_non_wav_exts = ('.m4a', '.mp3', '.flac', '.ogg', '.webm')
_to_convert = [
    f for f in sorted(_audio_dir_path.iterdir())
    if f.is_file()
    and f.name.lower().startswith('utterance_')
    and f.suffix.lower() in _non_wav_exts
]
if _to_convert:
    print(f'Preprocess: converting {len(_to_convert)} audio(s) to WAV 16 kHz mono s16...')
    for src in _to_convert:
        dst = src.with_suffix('.wav')
        cmd = [
            'ffmpeg', '-y', '-i', str(src),
            '-ar', '16000', '-ac', '1', '-sample_fmt', 's16',
            str(dst),
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  FAIL {src.name}: {r.stderr[-300:]}')
            continue
        shutil.move(str(src), str(_backup_dir / src.name))
        print(f'  OK   {src.name} -> {dst.name} ({dst.stat().st_size} B)')
else:
    print('Preprocess: all utterances are already WAV.')
print()

# Detect available utterances
# Tolerant detection: accept any common audio extension and find which utterances
# are present. Builds a stem -> filename mapping (e.g. 'utterance_03' -> 'utterance_03.wav').
_STEM_RE = re.compile(r'^(utterance_\d{2})\.([a-z0-9]+)$', re.IGNORECASE)
AUDIO_FILE_BY_STEM = {}
for fname in sorted(os.listdir(AUDIO_DIR)):
    m = _STEM_RE.match(fname)
    if not m:
        continue
    stem = m.group(1).lower()
    ext = m.group(2).lower()
    if f'.{ext}' not in AUDIO_EXTS:
        continue
    if stem in AUDIO_FILE_BY_STEM:
        prev = AUDIO_FILE_BY_STEM[stem]
        print(f'Warning: duplicate audio for {stem} -> keeping {prev}, ignoring {fname}')
        continue
    AUDIO_FILE_BY_STEM[stem] = fname

AVAILABLE_STEMS = sorted(AUDIO_FILE_BY_STEM.keys())
MISSING_STEMS = [s for s in EXPECTED_STEMS if s not in AUDIO_FILE_BY_STEM]

if not AVAILABLE_STEMS:
    raise FileNotFoundError(
        f'No utterances found in {AUDIO_DIR}. Expected files like utterance_01.wav or utterance_01.m4a'
    )

print(f'Utterances available: {len(AVAILABLE_STEMS)} / {len(EXPECTED_STEMS)}')
for stem in AVAILABLE_STEMS:
    print(f'  [OK] {AUDIO_FILE_BY_STEM[stem]}')

if MISSING_STEMS:
    print()
    print(f'Warning: {len(MISSING_STEMS)} utterances missing (will be skipped):')
    for stem in MISSING_STEMS:
        print(f'  [missing] {stem}.<ext>')
    print('Re-record and re-run when ready.')


### 1.2 Install dependencies

- `ffmpeg` - required by `openai-whisper` to load audio files
- `jiwer` - WER and CER computation
- `openai-whisper` - local Whisper inference
- `librosa` + `sentencepiece` - required by the HuggingFace ASR models

**Note:** we deliberately do NOT install `torch` because Colab provides it pre-installed with CUDA support. Reinstalling via `pip` would replace it with the CPU-only version.

In [ ]:
!apt-get install -y ffmpeg -qq > /dev/null
!pip install -q jiwer openai-whisper librosa sentencepiece


### 1.3 Imports and device check

Verifies that CUDA is available. If `Device: cpu` appears, switch the runtime type to GPU before continuing.

In [ ]:
import json
import re
import time
import gc
import random
import datetime
import platform
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

import torch
import whisper
from jiwer import wer as wer_fn, cer as cer_fn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Torch: {torch.__version__}')

# Reproducibility seeds. With temperature=0.0 pinned in the decoding protocol these
# are no-ops for transcription output, but they are standard hygiene for any
# downstream numpy/torch randomness and let us cite a single seed in the repro card.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Seed: {SEED}')

## 2. Configuration

Defines the dataset, the domain keywords, the models to evaluate, and the domain prompt used in the *with-prompt* condition.

### 2.1 Dataset - 30 phrases organized in 6 categories

Phrases are designed to cover the full real-use scope of the VisionLeaderUFV application:

| Category | Phrases | Purpose |
|-|-|-|
| `initial_location` | 5 | User describes where they are (with room codes) |
| `destination` | 5 | User states where they want to go |
| `navigation_command` | 5 | Commands during navigation (cancel, repeat, etc.) |
| `confirmation` | 5 | Yes/no/done responses during the flow |
| `hard_domain` | 5 | Long phrases combining several domain terms (stress test) |
| `edge_case` | 5 | Atypical syntax (open questions, polite forms, paraphrases) |


In [ ]:
PHRASES_ALL = [
    {'id': 'utterance_01', 'category': 'initial_location', 'ground_truth': 'I am in classroom 0.3 building G'},
    {'id': 'utterance_02', 'category': 'initial_location', 'ground_truth': 'I am in classroom 2.1 on the second floor'},
    {'id': 'utterance_03', 'category': 'initial_location', 'ground_truth': 'I am at the main entrance of building G'},
    {'id': 'utterance_04', 'category': 'initial_location', 'ground_truth': 'I am in the AI laboratory on the ground floor of building G'},
    {'id': 'utterance_05', 'category': 'initial_location', 'ground_truth': 'I am next to the elevator on floor 1 of building G'},
    {'id': 'utterance_06', 'category': 'destination', 'ground_truth': 'I want to go to classroom 1.4'},
    {'id': 'utterance_07', 'category': 'destination', 'ground_truth': 'Take me to the gym in building CD'},
    {'id': 'utterance_08', 'category': 'destination', 'ground_truth': 'I want to go to the cafeteria'},
    {'id': 'utterance_09', 'category': 'destination', 'ground_truth': 'Take me to the nearest restroom on this floor'},
    {'id': 'utterance_10', 'category': 'destination', 'ground_truth': 'Could you guide me to the EPS faculty offices please'},
    {'id': 'utterance_11', 'category': 'navigation_command', 'ground_truth': 'I am in a hurry please use the stairs'},
    {'id': 'utterance_12', 'category': 'navigation_command', 'ground_truth': 'Repeat the instructions please'},
    {'id': 'utterance_13', 'category': 'navigation_command', 'ground_truth': 'I am lost I need help'},
    {'id': 'utterance_14', 'category': 'navigation_command', 'ground_truth': 'Cancel navigation and start again'},
    {'id': 'utterance_15', 'category': 'navigation_command', 'ground_truth': 'Stop and tell me where I am'},
    {'id': 'utterance_16', 'category': 'confirmation', 'ground_truth': 'Yes I am ready to continue'},
    {'id': 'utterance_17', 'category': 'confirmation', 'ground_truth': 'No that is not my location'},
    {'id': 'utterance_18', 'category': 'confirmation', 'ground_truth': 'I have arrived at the elevator on floor 2'},
    {'id': 'utterance_19', 'category': 'confirmation', 'ground_truth': 'Done I have reached my destination'},
    {'id': 'utterance_20', 'category': 'confirmation', 'ground_truth': 'Yes I can see the elevator and the doors are open'},
    {'id': 'utterance_21', 'category': 'hard_domain', 'ground_truth': 'I am in classroom 1.1 and I want to go to the EPS faculty offices'},
    {'id': 'utterance_22', 'category': 'hard_domain', 'ground_truth': 'Take me from classroom 0.1 to the AI laboratory on the ground floor'},
    {'id': 'utterance_23', 'category': 'hard_domain', 'ground_truth': 'I just came out of the elevator on the first floor of building G'},
    {'id': 'utterance_24', 'category': 'hard_domain', 'ground_truth': 'I do not know where I am I need help finding my location'},
    {'id': 'utterance_25', 'category': 'hard_domain', 'ground_truth': 'I am at the directory sign in building G next to the mentoring room'},
    {'id': 'utterance_26', 'category': 'edge_case', 'ground_truth': 'How do I get from building CD to building G'},
    {'id': 'utterance_27', 'category': 'edge_case', 'ground_truth': 'Where is the secretary office in building G'},
    {'id': 'utterance_28', 'category': 'edge_case', 'ground_truth': 'I need to find classroom 0.3 quickly'},
    {'id': 'utterance_29', 'category': 'edge_case', 'ground_truth': 'Could you tell me how to reach the AI laboratory from here'},
    {'id': 'utterance_30', 'category': 'edge_case', 'ground_truth': 'My destination is classroom 1.4 in building G please'},
]

# Filter to only the phrases whose audio file is actually available,
# attaching the resolved filename so the inference loop works regardless of extension.
PHRASES = []
for p in PHRASES_ALL:
    fname = AUDIO_FILE_BY_STEM.get(p['id'])
    if fname is None:
        continue
    PHRASES.append({**p, 'audio': fname})

print(f'Loaded {len(PHRASES)} phrases (out of {len(PHRASES_ALL)} expected) across {len(set(p["category"] for p in PHRASES))} categories')


### 2.2 Domain keywords for DKA

Keywords whose correct transcription is critical for the navigation system to function (room codes, building codes, navigation elements, voice commands).

In [ ]:
DOMAIN_KEYWORDS = [
    # Room numbers
    '0.1', '0.3', '1.1', '1.4', '2.1',
    # Buildings
    'building g', 'building cd',
    # Navigation elements and rooms
    'elevator', 'stairs', 'cafeteria', 'ai laboratory', 'restroom',
    'faculty offices', 'main entrance', 'mentoring room',
    'secretary', 'directory sign', 'gym',
    # Floors
    'ground floor', 'first floor', 'second floor', 'floor 1', 'floor 2',
    # Voice commands and intents
    'cancel', 'repeat', 'hurry', 'lost', 'stop', 'done', 'destination',
    'doors are open',
]

print(f'Tracking {len(DOMAIN_KEYWORDS)} domain keywords')


### 2.3 Domain prompt

The same prompt the production app feeds to Whisper. It primes the model with the most likely vocabulary for this domain. The *with-prompt* condition uses this string; the *no-prompt* condition runs without any prompt context.

In [ ]:
# Domain prompt aligned strictly with the vocabulary present in the 30 test utterances.
# This isolates the experimental variable (prompt impact) by avoiding extra terms
# that don't appear in the dataset, which could otherwise bias the model in the
# with_prompt condition.
DOMAIN_PROMPT = (
    'UFV campus navigation assistant. '
    'classroom 0.1, classroom 0.3, classroom 1.1, classroom 1.4, classroom 2.1, '
    'building G, building CD, '
    'main entrance, ground floor, first floor, second floor, floor 1, floor 2, '
    'elevator, stairs, cafeteria, gym, restroom, '
    'AI laboratory, mentoring room, EPS faculty offices, secretary office, directory sign, '
    'I want to go to, take me to, I am in, I am at, I am next to, I am lost, '
    'yes, no, repeat, cancel, stop, done, '
    'I am in a hurry, I have arrived, my destination, the doors are open.'
)

print(f'Domain prompt: {len(DOMAIN_PROMPT)} chars, {len(DOMAIN_PROMPT.split())} words')


### 2.4 Models and conditions

Eight models: six Whisper checkpoints (tiny to large-v3) and two other ASR
architectures (Wav2Vec2 CTC, Speech2Text), all on the same Colab GPU. Whisper runs
under both prompt conditions; the HF models run once and are duplicated across
conditions (they take no prompt), so their prompt-impact delta is zero.

Each (model, condition) pair is timed several times after a per-model warmup.
Quality metrics come from the first run, since the pinned greedy decoding makes
repeated transcriptions identical.

In [ ]:
# All models run locally on Colab GPU for a fair comparison on identical hardware.
# Whisper models span the full size scale (tiny -> large-v3) plus two alternative
# ASR architectures (Wav2Vec2 CTC, Speech2Text Seq2Seq) as baselines from other families.
LOCAL_MODELS = [
    {'name': 'tiny', 'group': 'local', 'size': 'tiny'},
    {'name': 'base', 'group': 'local', 'size': 'base'},
    {'name': 'small', 'group': 'local', 'size': 'small'},
    {'name': 'medium', 'group': 'local', 'size': 'medium'},
    {'name': 'large-v3-turbo', 'group': 'local', 'size': 'large-v3-turbo'},
    {'name': 'large-v3', 'group': 'local', 'size': 'large-v3'},
]

# HuggingFace models from other ASR families.
# They do NOT accept a prompt parameter, so the with_prompt condition
# reports the same result as no_prompt for fair tabulation.
HF_MODELS = [
    {'name': 'facebook/wav2vec2-base-960h', 'group': 'hf', 'size': 'wav2vec2-base', 'arch': 'wav2vec2', 'supports_prompt': False},
    {'name': 'facebook/s2t-small-librispeech-asr', 'group': 'hf', 'size': 's2t-small', 'arch': 'speech2text', 'supports_prompt': False},
]

CONDITIONS = ['no_prompt', 'with_prompt']

# Latency benchmarking protocol:
#   - N_WARMUP inferences per model are discarded before timing (pays cold-start cost once:
#     CUDA kernel compilation, VRAM weight load, thermal/cache stabilization).
#   - Each (model, condition) is then timed N_LATENCY_RUNS times over the 30 audios.
#   - Reported avg_latency_s     = mean over all N_LATENCY_RUNS * 30 samples.
#   - Reported avg_latency_s_std = sample std across the N_LATENCY_RUNS per-run means
#     (measures inter-run stability, independent from cross-phrase variance).
#   - Quality (WER/CER/SentAcc/DKA) is computed once from run 1 predictions. With the
#     pinned decoding protocol (temperature=0.0, greedy, no fallback ladder) Whisper is
#     bit-deterministic across runs, so runs 2..N add no information beyond latency.
N_WARMUP = 3
N_LATENCY_RUNS = 3

n_models = len(LOCAL_MODELS) + len(HF_MODELS)
n_timed = (
    len(LOCAL_MODELS) * len(CONDITIONS) * N_LATENCY_RUNS * len(PHRASES)
    + len(HF_MODELS) * N_LATENCY_RUNS * len(PHRASES)
)
print(f'Models: {len(LOCAL_MODELS)} Whisper local + {len(HF_MODELS)} HF = {n_models} total')
print(f'Conditions: {CONDITIONS}')
print(f'Warmup iters per model: {N_WARMUP}')
print(f'Latency runs per (model, condition): {N_LATENCY_RUNS}')
print(f'Timed inferences expected: {n_timed} (+ {n_models * N_WARMUP} warmup, discarded)')

### 2.5 Output paths

All results are stored under `/content/results/whisper/`. Each model gets its own subdirectory with the per-row CSV.

In [ ]:
RESULTS_PATH = '/content/results/whisper'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Synthetic warmup audio: 5 seconds of silence at 16 kHz mono 16-bit PCM.
# Byte-identical across runs (all zeros, no FP math) and disjoint from the 30
# measured utterances, so no phrase benefits from a pre-warm disk/memory cache
# on its first measurement.
from scipy.io import wavfile
WARMUP_AUDIO_PATH = '/content/warmup_synthetic.wav'
if not os.path.exists(WARMUP_AUDIO_PATH):
    _pcm = np.zeros(16000 * 5, dtype=np.int16)
    wavfile.write(WARMUP_AUDIO_PATH, 16000, _pcm)

print(f'Results will be saved to {RESULTS_PATH}')
print(f'Warmup audio: {WARMUP_AUDIO_PATH}')

## 3. Reproducibility card

Captures the environment (Python, PyTorch, GPU, git commit, random seed), the pinned decoding and latency protocols, and SHA-256 fingerprints of the 30 input audios. The resulting JSON answers every "how was this configured?" question without grepping the notebook.

In [ ]:
def get_git_commit():
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', '--short', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return 'N/A'

def _file_sha256(path, chunk_size=65536):
    """SHA-256 of a file, streamed in chunks so large audio files do not spike RAM."""
    import hashlib
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

# Per-audio SHA-256 fingerprints so anyone re-running the experiment can verify
# that they are using the identical dataset bytes. Computed once, stored in the card.
dataset_audio_hashes = {}
for phrase in PHRASES:
    audio_path = os.path.join(AUDIO_DIR, phrase['audio'])
    dataset_audio_hashes[phrase['audio']] = {
        'utterance_id': phrase['id'],
        'sha256': _file_sha256(audio_path),
        'size_bytes': os.path.getsize(audio_path),
    }

warmup_audio_file = WARMUP_AUDIO_PATH
warmup_audio_sha256 = (
    _file_sha256(WARMUP_AUDIO_PATH) if os.path.exists(WARMUP_AUDIO_PATH) else None
)

# Pinned decoding protocol (kept in sync with transcribe_local / transcribe_hf below).
# Any tribunal question of the form "how did you fix decoding for reproducibility?"
# is answered from this JSON alone without grep-ing the notebook.
DECODING_PROTOCOL_WHISPER = {
    'temperature': 0.0,
    'beam_size': None,
    'best_of': None,
    'task': 'transcribe',
    'language': 'en',
    'condition_on_previous_text': False,
    'fp16': bool(torch.cuda.is_available()),
}
DECODING_PROTOCOL_HF = {
    'speech2text': {'num_beams': 1, 'do_sample': False},
    'wav2vec2': 'CTC argmax (deterministic by construction)',
}
LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_audio_file': warmup_audio_file,
    'warmup_audio_sha256': warmup_audio_sha256,
    'warmup_audio_source': 'synthetic 5-second silence, 16 kHz mono 16-bit PCM, disjoint from the 30 measured utterances',
    'phrase_order': 'deterministically shuffled per latency run (seed = SEED * 1000 + run_idx), identical across re-executions',
    'reports': 'mean over all N_LATENCY_RUNS * 30 samples; std across N_LATENCY_RUNS run-means',
}

repro = {
    'experiment': 'whisper_stt_evaluation',
    'date': datetime.datetime.now().isoformat(),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'git_commit': get_git_commit(),
    'seed': SEED,
    'local_models': [m['name'] for m in LOCAL_MODELS],
    'hf_models': [m['name'] for m in HF_MODELS],
    'conditions': CONDITIONS,
    'n_phrases': len(PHRASES),
    'n_keywords': len(DOMAIN_KEYWORDS),
    'domain_prompt_length_chars': len(DOMAIN_PROMPT),
    'domain_prompt': DOMAIN_PROMPT,
    'decoding_protocol_whisper': DECODING_PROTOCOL_WHISPER,
    'decoding_protocol_hf': DECODING_PROTOCOL_HF,
    'latency_protocol': LATENCY_PROTOCOL,
    'dataset_audio_hashes': dataset_audio_hashes,
}

repro_path = f'{RESULTS_PATH}/whisper_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)
print(f'Reproducibility card saved to {repro_path}')
# Print a compact view without the hash table (would be 30 lines of hex).
repro_preview = {k: v for k, v in repro.items() if k != 'dataset_audio_hashes'}
repro_preview['dataset_audio_hashes'] = f'<{len(dataset_audio_hashes)} SHA-256 fingerprints saved to JSON>'
print(json.dumps(repro_preview, indent=2))

## 4. Helper functions

Text normalization, metric computation, and inference wrappers. Splitting helpers across small cells keeps each function focused and easy to inspect.

### 4.1 Text normalization

Two normalizers:

- `normalize_text` strips incidental punctuation (commas, semicolons, trailing dots) but preserves decimal dots in numbers like `0.3` and `1.4`. Used for `Sentence Accuracy` and as input to WER/CER.
- `normalize_for_dka` additionally rewrites spelled-out numbers (`zero point three`) into their numeric form (`0.3`). This avoids penalizing models that dictate numbers as words.

In [ ]:
_PUNCT_RE = re.compile(r"[,;:!?\"\(\)]")
_TRAIL_DOT_RE = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WS_RE = re.compile(r"\s+")

_SPOKEN_NUMS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
}

def normalize_text(text):
    """Lowercase, strip incidental punctuation, collapse whitespace.
    Preserves dots inside decimal numbers (0.3, 1.4).
    """
    if text is None:
        return ''
    t = str(text).lower().strip()
    t = _PUNCT_RE.sub(' ', t)
    t = _TRAIL_DOT_RE.sub('', t)
    t = _WS_RE.sub(' ', t).strip()
    return t

def normalize_for_dka(text):
    """Same as normalize_text but additionally rewrites 'zero point three' -> '0.3'.
    Used for Domain Keyword Accuracy comparisons.
    """
    t = normalize_text(text)
    pattern = re.compile(r'\b(' + '|'.join(_SPOKEN_NUMS) + r')\s+point\s+(' + '|'.join(_SPOKEN_NUMS) + r')\b')
    def repl(m):
        return f"{_SPOKEN_NUMS[m.group(1)]}.{_SPOKEN_NUMS[m.group(2)]}"
    return pattern.sub(repl, t)

### 4.2 Per-phrase metrics

WER and CER come from `jiwer` and operate on normalized text. `Sentence Accuracy` is a strict equality check after normalization. `DKA` returns `None` when the ground truth contains no tracked keywords (so phrases without domain content do not pull the average up or down).

In [ ]:
def safe_wer(gt, pred):
    if not gt:
        return 0.0
    return float(wer_fn(normalize_text(gt), normalize_text(pred)))

def safe_cer(gt, pred):
    if not gt:
        return 0.0
    return float(cer_fn(normalize_text(gt), normalize_text(pred)))

def compute_sentence_accuracy(prediction, ground_truth):
    """1.0 if normalized prediction equals normalized ground truth; 0.0 otherwise."""
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0

def compute_dka(prediction, ground_truth, keywords):
    """Fraction of ground-truth domain keywords that also appear in the prediction.
    Returns None if no tracked keyword is present in the ground truth.
    """
    pred_norm = normalize_for_dka(prediction)
    gt_norm = normalize_for_dka(ground_truth)
    gt_keywords = [kw for kw in keywords if kw in gt_norm]
    if not gt_keywords:
        return None
    found = sum(1 for kw in gt_keywords if kw in pred_norm)
    return found / len(gt_keywords)

### 4.3 Aggregation helper

Reduces a list of per-phrase row dicts to a single dict of mean metrics. `DKA` is averaged only over phrases where it is defined (i.e. that contain at least one tracked keyword).

In [ ]:
def aggregate_results(rows, run_latency_means=None):
    """Reduce per-phrase rows to a dict of mean metrics.

    If `run_latency_means` is provided (one entry per timed latency run),
    `avg_latency_s_std` is the sample std across those run means - a measure of
    inter-run stability that is independent from cross-phrase variance.
    """
    df = pd.DataFrame(rows)
    dka_vals = df['dka'].dropna()
    result = {
        'wer_pct': round(df['wer'].mean() * 100, 2),
        'cer_pct': round(df['cer'].mean() * 100, 2),
        'sentence_accuracy_pct': round(df['sentence_accuracy'].mean() * 100, 2),
        'dka_pct': round(dka_vals.mean() * 100, 2) if len(dka_vals) > 0 else None,
        'avg_latency_s': round(df['latency_s'].mean(), 3),
        'n_phrases': len(df),
    }
    if run_latency_means is not None and len(run_latency_means) >= 2:
        result['avg_latency_s_std'] = round(float(np.std(run_latency_means, ddof=1)), 4)
    else:
        result['avg_latency_s_std'] = 0.0
    return result

def safe_model_name(name):
    return name.replace('/', '_').replace('-', '_')

### 4.4 Transcription wrappers

Two thin wrappers (one per backend) that both return `(text, latency_seconds)` so the inference loop stays identical regardless of backend. Each wrapper:

- Surrounds the timed region with `torch.cuda.synchronize()` so `time.time()` reflects actual GPU work, not just kernel enqueue.
- Pins the decoding protocol declared in section 3: `temperature=0.0` (greedy, no fallback ladder), `condition_on_previous_text=False`, `fp16=True` on GPU for Whisper; `num_beams=1, do_sample=False` for HuggingFace Seq2Seq. Wav2Vec2 is CTC argmax and therefore deterministic by construction.

In [ ]:
def _sync():
    """Force the CUDA stream to drain so time.time() reflects actual GPU work,
    not just kernel enqueue. No-op on CPU.
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Pinned Whisper decoding kwargs. Crucially, temperature is a float (not a tuple),
# which disables Whisper's default fallback ladder (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
# and forces pure greedy decoding with no stochastic escalation on hard audios.
_WHISPER_DECODING_KWARGS = {
    'language': 'en',
    'task': 'transcribe',
    'temperature': 0.0,
    'beam_size': None,
    'best_of': None,
    'condition_on_previous_text': False,
}

def transcribe_local(model, audio_path, prompt=None):
    """Transcribe a single audio with a local openai-whisper model. Returns (text, latency_s)."""
    kwargs = dict(_WHISPER_DECODING_KWARGS)
    kwargs['fp16'] = torch.cuda.is_available()
    if prompt:
        kwargs['initial_prompt'] = prompt
    _sync()
    t0 = time.time()
    result = model.transcribe(audio_path, **kwargs)
    _sync()
    latency = time.time() - t0
    return result['text'].strip(), latency

def transcribe_hf(model, processor, arch, audio_path):
    """Transcribe with a HuggingFace ASR model (wav2vec2 or speech2text).
    These architectures do not accept a prompt parameter. Returns (text, latency_s).
    """
    import librosa
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt').to(DEVICE)
    _sync()
    t0 = time.time()
    with torch.no_grad():
        if arch == 'wav2vec2':
            # CTC argmax - deterministic by construction, no decoding params to pin.
            logits = model(inputs.input_values).logits
            predicted_ids = torch.argmax(logits, dim=-1)
            text = processor.batch_decode(predicted_ids)[0]
        elif arch == 'speech2text':
            # Explicit greedy on model.generate() to match Whisper's pinned protocol.
            generated_ids = model.generate(
                inputs.input_features,
                attention_mask=inputs.get('attention_mask'),
                num_beams=1,
                do_sample=False,
            )
            text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        else:
            raise ValueError(f'Unknown HF arch: {arch}')
    _sync()
    latency = time.time() - t0
    return text.strip(), latency

### 4.5 Warmup helpers

Per-model warmup routines that run `N_WARMUP` inferences on a real dataset audio and discard their timings. They pay each model's cold-start cost (CUDA kernel compilation, VRAM weight load, cache and thermal stabilization) outside the timed region so the first measured run is not contaminated.

In [ ]:
def warmup_local(model, audio_path, n=None):
    """Run n Whisper inferences and discard timings."""
    n = n if n is not None else N_WARMUP
    for _ in range(n):
        transcribe_local(model, audio_path)

def warmup_hf(model, processor, arch, audio_path, n=None):
    """Run n HuggingFace ASR inferences and discard timings."""
    n = n if n is not None else N_WARMUP
    for _ in range(n):
        transcribe_hf(model, processor, arch, audio_path)

### 4.6 Sanity checks

Tiny smoke test on the helpers before running the full experiment, so any normalization regression fails immediately rather than at aggregation time.

In [ ]:
assert normalize_text('Hello, world.') == 'hello world'
assert normalize_text('Classroom 0.3.') == 'classroom 0.3'
assert normalize_text('Building C.D.') == 'building cd'  # acronyms with dots collapse
assert normalize_text('U.S.A.') == 'usa'
assert compute_sentence_accuracy('Hello, world.', 'hello world') == 1.0
assert compute_sentence_accuracy('Building C.D.', 'building cd') == 1.0
assert compute_dka('I am in classroom 0.3', 'I am in classroom 0.3 building G', DOMAIN_KEYWORDS) == 0.5
assert normalize_for_dka('zero point three') == '0.3'
print('Sanity checks passed.')


## 5. Run experiments

Two independent loops, one per inference backend. For each model:

1. Load weights and warm up (`N_WARMUP` discarded inferences on a synthetic silent WAV, disjoint from the measured set).
2. For every `(condition, phrase)` pair, time `N_LATENCY_RUNS` sequential passes over the 30 audios. The phrase order is shuffled deterministically per run (seeded from `SEED * 1000 + run_idx`) so thermal drift does not systematically bias any specific phrase position.
3. Save per-phrase rows with predictions, quality metrics, and per-run latencies; append a per-model-condition aggregate to `all_summaries`.

### 5.1 Local Whisper models on Colab GPU

The six Whisper checkpoints, each under both prompt conditions, over the 30
utterances and the repeated latency runs. Models are released from VRAM between
iterations to avoid running out of memory.

In [ ]:
all_summaries = []

for meta in LOCAL_MODELS:
    model_name = meta['name']
    safe = safe_model_name(f'whisper-{model_name}')
    model_dir = f'{RESULTS_PATH}/{safe}'
    os.makedirs(model_dir, exist_ok=True)

    print()
    print('=' * 60)
    print(f'LOCAL: whisper-{model_name}')
    print('=' * 60)

    t0 = time.time()
    print('  Loading model...')
    local_model = whisper.load_model(model_name, device=DEVICE)
    print(f'  Loaded in {time.time() - t0:.1f}s')

    # Warmup on a synthetic silent WAV - disjoint from the measured set so no
    # phrase benefits from a pre-warm disk cache on its first timed run.
    print(f'  Warming up ({N_WARMUP} iters on {WARMUP_AUDIO_PATH}, discarded)...')
    warmup_local(local_model, WARMUP_AUDIO_PATH)

    rows_all = []
    for condition in CONDITIONS:
        prompt_to_use = DOMAIN_PROMPT if condition == 'with_prompt' else None
        print(f'  -> condition: {condition} ({N_LATENCY_RUNS} timed runs, phrase order shuffled per run)')

        # Run 1 collects predictions AND latencies; runs 2..N_LATENCY_RUNS collect
        # only latencies. Under the pinned decoding protocol (temperature=0.0, greedy,
        # no fallback ladder) the predictions are bit-identical across runs, so
        # additional prediction passes would carry no information.
        predictions = {}
        per_phrase_latencies = {p['id']: [] for p in PHRASES}
        run_means = []

        for run_idx in range(N_LATENCY_RUNS):
            # Deterministic per-run shuffle: identical across re-executions (same seed)
            # but different across runs, averaging out any thermal drift across phrase
            # positions rather than letting it pile onto the same frames every time.
            _run_rng = np.random.default_rng(SEED * 1000 + run_idx)
            phrase_order = list(PHRASES)
            _run_rng.shuffle(phrase_order)

            run_lats = []
            for phrase in phrase_order:
                audio_path = os.path.join(AUDIO_DIR, phrase['audio'])
                try:
                    pred, latency = transcribe_local(local_model, audio_path, prompt_to_use)
                except Exception as e:
                    print(f'     [FAIL] run {run_idx + 1} {phrase["id"]}: {e}')
                    pred, latency = '', 0.0
                per_phrase_latencies[phrase['id']].append(latency)
                run_lats.append(latency)
                if run_idx == 0:
                    predictions[phrase['id']] = pred
            run_means.append(float(np.mean(run_lats)))

        # Build per-phrase rows (in stable PHRASES order for CSV consistency)
        # with aggregated and per-run latency columns.
        rows = []
        for phrase in PHRASES:
            gt = phrase['ground_truth']
            pred = predictions.get(phrase['id'], '')
            lats = per_phrase_latencies[phrase['id']]
            lat_mean = float(np.mean(lats)) if lats else 0.0
            lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
            row = {
                'model': f'whisper-{model_name}',
                'group': 'local',
                'condition': condition,
                'phrase_id': phrase['id'],
                'category': phrase['category'],
                'ground_truth': gt,
                'prediction': pred,
                'wer': safe_wer(gt, pred),
                'cer': safe_cer(gt, pred),
                'sentence_accuracy': compute_sentence_accuracy(pred, gt),
                'dka': compute_dka(pred, gt, DOMAIN_KEYWORDS),
                'latency_s': round(lat_mean, 4),
                'latency_s_std': round(lat_std, 4),
            }
            for i, lat in enumerate(lats, start=1):
                row[f'latency_s_run{i}'] = round(lat, 4)
            rows.append(row)
            rows_all.append(row)

        agg = aggregate_results(rows, run_latency_means=run_means)
        agg.update({
            'model': f'whisper-{model_name}', 'group': 'local', 'size': meta['size'],
            'condition': condition, 'failed': 0,
        })
        all_summaries.append(agg)
        print('     WER={:.2f}% CER={:.2f}% SentAcc={:.2f}% DKA={}% Lat={:.3f}s +/- {:.3f}s'.format(
            agg['wer_pct'], agg['cer_pct'], agg['sentence_accuracy_pct'], agg['dka_pct'],
            agg['avg_latency_s'], agg['avg_latency_s_std']))

    pd.DataFrame(rows_all).to_csv(f'{model_dir}/results.csv', index=False)
    print(f'  Saved: {model_dir}/results.csv ({len(rows_all)} rows)')

    del local_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
total_done = len(LOCAL_MODELS) * len(CONDITIONS) * N_LATENCY_RUNS * len(PHRASES)
print(f'Local experiment complete. {total_done} timed inferences.')

### 5.2 HuggingFace ASR models (Wav2Vec2 and Speech2Text)

Two models from other ASR families to contrast against the Whisper line.

- `facebook/wav2vec2-base-960h` - CTC-based ASR, English-only, trained on 960h LibriSpeech
- `facebook/s2t-small-librispeech-asr` - Sequence-to-sequence ASR, English-only

**Important:** these architectures do not accept a prompt parameter. For consistency with the rest of the experiment, the same transcription is reported under both `no_prompt` and `with_prompt` conditions, which makes the prompt-impact delta exactly zero (a valid finding: prompt steering does not generalize beyond Whisper).

Each HF model is still run `N_LATENCY_RUNS` times after warmup so its reported latency carries the same inter-run std as the Whisper line.

In [ ]:
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    Speech2TextForConditionalGeneration, Speech2TextProcessor,
)

for meta in HF_MODELS:
    model_name = meta['name']
    safe = safe_model_name(model_name)
    model_dir = f'{RESULTS_PATH}/{safe}'
    os.makedirs(model_dir, exist_ok=True)

    print()
    print('=' * 60)
    print(f'HF: {model_name}')
    print('=' * 60)

    t0 = time.time()
    print('  Loading model...')
    if meta['arch'] == 'wav2vec2':
        processor = Wav2Vec2Processor.from_pretrained(model_name)
        hf_model = Wav2Vec2ForCTC.from_pretrained(model_name).to(DEVICE).eval()
    elif meta['arch'] == 'speech2text':
        processor = Speech2TextProcessor.from_pretrained(model_name)
        hf_model = Speech2TextForConditionalGeneration.from_pretrained(model_name).to(DEVICE).eval()
    else:
        raise ValueError(f'Unknown arch: {meta["arch"]}')
    print(f'  Loaded in {time.time() - t0:.1f}s')

    # Warmup on the same synthetic WAV used by the Whisper loop.
    print(f'  Warming up ({N_WARMUP} iters on {WARMUP_AUDIO_PATH}, discarded)...')
    warmup_hf(hf_model, processor, meta['arch'], WARMUP_AUDIO_PATH)

    print(f'  -> running inference ({N_LATENCY_RUNS} timed runs, phrase order shuffled per run, prompt not supported)')
    # Wav2Vec2 is CTC argmax (deterministic by construction) and Speech2Text is pinned
    # to greedy (num_beams=1, do_sample=False) in transcribe_hf, so runs 2..N add
    # nothing beyond latency measurement.
    predictions = {}
    per_phrase_latencies = {p['id']: [] for p in PHRASES}
    run_means = []
    failed = 0

    for run_idx in range(N_LATENCY_RUNS):
        _run_rng = np.random.default_rng(SEED * 1000 + run_idx)
        phrase_order = list(PHRASES)
        _run_rng.shuffle(phrase_order)

        run_lats = []
        for phrase in phrase_order:
            audio_path = os.path.join(AUDIO_DIR, phrase['audio'])
            try:
                pred, latency = transcribe_hf(hf_model, processor, meta['arch'], audio_path)
            except Exception as e:
                print(f'     [FAIL] run {run_idx + 1} {phrase["id"]}: {e}')
                pred, latency = '', 0.0
                if run_idx == 0:
                    failed += 1
            per_phrase_latencies[phrase['id']].append(latency)
            run_lats.append(latency)
            if run_idx == 0:
                predictions[phrase['id']] = pred
        run_means.append(float(np.mean(run_lats)))

    rows_all = []
    rows_single = []
    for phrase in PHRASES:
        gt = phrase['ground_truth']
        pred = predictions.get(phrase['id'], '')
        lats = per_phrase_latencies[phrase['id']]
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
        # Same prediction and latency stats duplicated across both conditions
        # (HF models ignore the prompt parameter).
        for condition in CONDITIONS:
            row = {
                'model': model_name,
                'group': 'hf',
                'condition': condition,
                'phrase_id': phrase['id'],
                'category': phrase['category'],
                'ground_truth': gt,
                'prediction': pred,
                'wer': safe_wer(gt, pred),
                'cer': safe_cer(gt, pred),
                'sentence_accuracy': compute_sentence_accuracy(pred, gt),
                'dka': compute_dka(pred, gt, DOMAIN_KEYWORDS),
                'latency_s': round(lat_mean, 4),
                'latency_s_std': round(lat_std, 4),
            }
            for i, lat in enumerate(lats, start=1):
                row[f'latency_s_run{i}'] = round(lat, 4)
            rows_all.append(row)
            if condition == 'no_prompt':
                rows_single.append(row)

    # Aggregate once (rows_single = no_prompt pass) and duplicate for both conditions.
    for condition in CONDITIONS:
        agg = aggregate_results(rows_single, run_latency_means=run_means)
        agg.update({
            'model': model_name, 'group': 'hf', 'size': meta['size'],
            'condition': condition, 'failed': failed,
        })
        all_summaries.append(agg)
        print('     [{}] WER={:.2f}% CER={:.2f}% SentAcc={:.2f}% DKA={}% Lat={:.3f}s +/- {:.3f}s'.format(
            condition, agg['wer_pct'], agg['cer_pct'], agg['sentence_accuracy_pct'], agg['dka_pct'],
            agg['avg_latency_s'], agg['avg_latency_s_std']))

    pd.DataFrame(rows_all).to_csv(f'{model_dir}/results.csv', index=False)
    print(f'  Saved: {model_dir}/results.csv ({len(rows_all)} rows)')

    del hf_model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
total_done = len(HF_MODELS) * N_LATENCY_RUNS * len(PHRASES)
print(f'HF experiment complete. {total_done} timed inferences ({N_LATENCY_RUNS} runs x {len(PHRASES)} phrases x {len(HF_MODELS)} models).')

## 6. Build summary tables

Two tables are produced from the in-memory `all_summaries` list and saved alongside the per-model CSVs.

### 6.1 Long table (one row per model x condition)

The natural format for downstream pivoting and plotting.

In [ ]:
summary_df = pd.DataFrame(all_summaries)
cols_order = [
    'model', 'group', 'size', 'condition',
    'wer_pct', 'cer_pct', 'sentence_accuracy_pct', 'dka_pct',
    'avg_latency_s', 'avg_latency_s_std', 'n_phrases', 'failed',
]
summary_df = summary_df[[c for c in cols_order if c in summary_df.columns]]
summary_df = summary_df.sort_values(['group', 'model', 'condition']).reset_index(drop=True)

summary_df.to_csv(f'{RESULTS_PATH}/all_models_summary_long.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary_long.csv')
print()
print(summary_df.to_string(index=False))

### 6.2 Wide table with prompt-impact deltas

One row per model with `no_prompt` and `with_prompt` side by side, plus the deltas.
This is the table for the thesis argument about the prompt's contribution.

In [ ]:
wide_rows = []
for model_name in summary_df['model'].unique():
    sub = summary_df[summary_df['model'] == model_name]
    np_ = sub[sub['condition'] == 'no_prompt']
    wp = sub[sub['condition'] == 'with_prompt']
    if np_.empty or wp.empty:
        continue
    np_, wp = np_.iloc[0], wp.iloc[0]
    wide_rows.append({
        'model': model_name,
        'group': np_['group'],
        'size': np_['size'],
        'wer_no_prompt': np_['wer_pct'],
        'wer_with_prompt': wp['wer_pct'],
        'wer_delta': round(np_['wer_pct'] - wp['wer_pct'], 2),
        'cer_no_prompt': np_['cer_pct'],
        'cer_with_prompt': wp['cer_pct'],
        'cer_delta': round(np_['cer_pct'] - wp['cer_pct'], 2),
        'sentence_acc_no_prompt': np_['sentence_accuracy_pct'],
        'sentence_acc_with_prompt': wp['sentence_accuracy_pct'],
        'dka_no_prompt': np_['dka_pct'],
        'dka_with_prompt': wp['dka_pct'],
        'dka_delta': round((wp['dka_pct'] or 0) - (np_['dka_pct'] or 0), 2),
        'latency_no_prompt_s': np_['avg_latency_s'],
        'latency_no_prompt_std_s': np_['avg_latency_s_std'],
        'latency_with_prompt_s': wp['avg_latency_s'],
        'latency_with_prompt_std_s': wp['avg_latency_s_std'],
    })
wide_df = pd.DataFrame(wide_rows)
wide_df.to_csv(f'{RESULTS_PATH}/all_models_summary_wide.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary_wide.csv')
print()
print(wide_df.to_string(index=False))

### 6.3 Per-utterance comparison table

For each utterance, shows the ground truth alongside the transcription produced by every model x condition combination. This is the qualitative view that lets you inspect *what each model heard* phrase by phrase, instead of just the aggregated metrics.

Saved as `per_utterance_predictions.csv`.

In [ ]:
# Build a wide per-utterance table: rows = utterances, columns = each model x condition prediction
all_rows = []
for safe_dir in os.listdir(RESULTS_PATH):
    p = os.path.join(RESULTS_PATH, safe_dir, 'results.csv')
    if os.path.isfile(p):
        all_rows.append(pd.read_csv(p))

if not all_rows:
    print('No per-model CSVs found. Skipping per-utterance view.')
else:
    full_df = pd.concat(all_rows, ignore_index=True)

    # Pivot: rows = phrase_id, one column per (model, condition)
    full_df['col'] = full_df['model'] + ' [' + full_df['condition'] + ']'
    pivot = full_df.pivot_table(
        index=['phrase_id', 'category', 'ground_truth'],
        columns='col',
        values='prediction',
        aggfunc='first',
    ).reset_index()

    pivot.to_csv(f'{RESULTS_PATH}/per_utterance_predictions.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/per_utterance_predictions.csv')
    print(f'Shape: {pivot.shape[0]} utterances x {pivot.shape[1] - 3} model-condition columns')
    print()

    # Show each utterance with its predictions in a vertical, readable format
    pred_cols = [c for c in pivot.columns if c not in {"phrase_id", "category", "ground_truth"}]
    for _, row in pivot.iterrows():
        print('=' * 80)
        print(f'{row["phrase_id"]}  [{row["category"]}]')
        print(f'  GROUND TRUTH: {row["ground_truth"]}')
        print('  -')
        for col in pred_cols:
            pred = row[col]
            if pd.isna(pred):
                pred = "(missing)"
            print(f'  {col}:')
            print(f'    {pred}')
        print()


### 6.4 Prompt Impact Score (PIS)

Names the prompt-impact deltas as a model characterisation metric:
`PIS_DKA = DKA_with - DKA_no` and `PIS_WER = WER_no - WER_with`. A high PIS means the
model leans on the prompt to recover domain vocabulary; the HF models score 0 by
design.

In [ ]:
pis_df = wide_df[[
    'model', 'group', 'size',
    'dka_no_prompt', 'dka_with_prompt', 'dka_delta',
    'wer_no_prompt', 'wer_with_prompt', 'wer_delta',
]].copy()
pis_df = pis_df.rename(columns={'dka_delta': 'pis_dka', 'wer_delta': 'pis_wer'})
pis_df = pis_df.sort_values('pis_dka', ascending=False).reset_index(drop=True)
pis_df.to_csv(f'{RESULTS_PATH}/prompt_impact_score.csv', index=False)
print(f'Saved: {RESULTS_PATH}/prompt_impact_score.csv')
print()
print('=== Prompt Impact Score (sorted by PIS_DKA) ===')
print(pis_df.to_string(index=False))


## 7. Utterance difficulty

Aggregates WER, sentence accuracy and DKA across all models and both conditions per
utterance, then ranks them hardest to easiest. The hardest ones are the natural
source of qualitative examples for the discussion. Saved as
`utterance_difficulty.csv`.

In [ ]:
if full_df.empty:
    print('No per-model CSVs available for utterance difficulty analysis.')
else:
    diff_rows = []
    for phrase_id in full_df['phrase_id'].unique():
        sub = full_df[full_df['phrase_id'] == phrase_id]
        diff_rows.append({
            'phrase_id': phrase_id,
            'category': sub['category'].iloc[0],
            'ground_truth': sub['ground_truth'].iloc[0],
            'mean_wer_pct': round(sub['wer'].mean() * 100, 2),
            'max_wer_pct': round(sub['wer'].max() * 100, 2),
            'mean_sentence_accuracy_pct': round(sub['sentence_accuracy'].mean() * 100, 2),
            'mean_dka_pct': round(sub['dka'].dropna().mean() * 100, 2) if sub['dka'].dropna().size else None,
            'n_runs': len(sub),
        })

    diff_df = pd.DataFrame(diff_rows)
    diff_df = diff_df.sort_values('mean_wer_pct', ascending=False).reset_index(drop=True)
    diff_df.to_csv(f'{RESULTS_PATH}/utterance_difficulty.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/utterance_difficulty.csv')
    print()
    print('=== Utterance difficulty (hardest first) ===')
    print(diff_df.to_string(index=False))


## 8. Final ranking and downloads

### 8.1 Ranking by DKA in the *with-prompt* condition

DKA in the with-prompt condition is the metric that best reflects real-world utility for the navigation domain: it scores how well each model preserves the vocabulary required by the downstream system, when given the domain context that production already provides.

In [ ]:
ranking_cols = ['model', 'group', 'wer_pct', 'cer_pct', 'sentence_accuracy_pct', 'dka_pct', 'avg_latency_s']
wp = summary_df[summary_df['condition'] == 'with_prompt'].sort_values('dka_pct', ascending=False)
print('FINAL RANKING (with prompt, sorted by DKA)')
print('=' * 70)
print(wp[ranking_cols].to_string(index=False))

### 8.2 Prompt impact summary

`wer_delta = wer_no_prompt - wer_with_prompt` (positive = prompt reduced WER).

`dka_delta = dka_with_prompt - dka_no_prompt` (positive = prompt improved DKA).

In [ ]:
print('PROMPT IMPACT (positive = prompt helped)')
print('=' * 70)
print(wide_df[['model', 'wer_delta', 'dka_delta']].to_string(index=False))

### 8.3 Output file inventory

In [ ]:
print('Output files:')
print(f'  {RESULTS_PATH}/whisper_reproducibility.json   (environment, protocols, audio hashes)')
print(f'  {RESULTS_PATH}/all_models_summary_long.csv     (one row per model and condition)')
print(f'  {RESULTS_PATH}/all_models_summary_wide.csv     (no_prompt vs with_prompt side by side)')
print(f'  {RESULTS_PATH}/per_utterance_predictions.csv   (ground truth plus every model prediction)')
print(f'  {RESULTS_PATH}/prompt_impact_score.csv         (models ranked by PIS_DKA)')
print(f'  {RESULTS_PATH}/utterance_difficulty.csv        (utterances ranked hardest to easiest)')
print(f'  {RESULTS_PATH}/<model>/results.csv             (per-model rows with per-run latencies)')

### 8.4 Download the full results folder

Run the next cell once the experiment has finished to bundle everything into a single `.zip` and download it to your local machine.

In [ ]:
from google.colab import files

!zip -r /content/whisper_results.zip /content/results/whisper > /dev/null
files.download('/content/whisper_results.zip')